# 🍜 STEP 05 — Process Framework: HITL + Sign-off Agent

전체 아키텍처의 핵심: **최종 검토 에이전트(Sign-off Agent)**의
**PASS / RETURN** 흐름을 Process Framework로 구현합니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| `ProcessBuilder` | 단계(Step)를 연결해 워크플로우 구성 |
| `KernelProcessStep` | 각 처리 단계 구현 |
| `context.emit_event()` | 단계 완료 시 다음 단계로 이벤트 전달 |
| HITL 게이트 | 특정 단계에서 인간 승인을 기다리고 재개 |
| Sign-off PASS/RETURN | 최종 검토 후 통과 또는 재작업 요청 |

---

## F&B 시스템 프로세스 흐름

```
Start
  ↓
[Step 1] ContextGatherStep     — 업종·위치·규모·단계·고민 수집
  ↓ ContextGathered
[Step 2] AgentRoutingStep      — 요청 유형 분류 → 전문 에이전트 호출
  ↓ ResultReady
[Step 3] DocumentGenerationStep — 결과를 문서로 변환 (영업신고서 초안 등)
  ↓ DocumentGenerated
[Step 4] HumanReviewStep ← HITL — 사용자 확인 (승인/거부)
  ↓ Approved               ↓ Rejected
[Step 5] SignOffStep      [Step 2로 RETURN]
  ↓
PASS → 최종 응답 반환
```

In [1]:
import asyncio, os, json
from enum import Enum
from typing import ClassVar
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function
from semantic_kernel.processes import ProcessBuilder
from semantic_kernel.processes.kernel_process import (
    KernelProcessStep,
    KernelProcessStepContext,
    KernelProcessStepState,
)
from semantic_kernel.processes.kernel_process.kernel_process_event import KernelProcessEvent
from semantic_kernel.processes.local_runtime.local_kernel_process import start

load_dotenv()

def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    ))
    return k

print("임포트 완료")

임포트 완료


---
## 1. 이벤트 정의

> 각 단계(Step)가 완료 시 emit할 이벤트를 Enum으로 정의합니다.
> 이벤트 이름이 `ProcessBuilder`에서 단계 간 연결의 키가 됩니다.

In [2]:
class FnbProcessEvent(str, Enum):
    """
    F&B 창업 지원 프로세스의 이벤트 목록.
    각 이벤트가 다음 단계로의 전환을 트리거합니다.
    """
    # ── 프로세스 진입점 ──
    Start                = "Start"                 # 사용자 최초 요청

    # ── Step 1 → Step 2 ──
    ContextGathered      = "ContextGathered"       # 컨텍스트 수집 완료

    # ── Step 2 → Step 3 ──
    AgentResultReady     = "AgentResultReady"      # 전문 에이전트 처리 완료

    # ── Step 3 → Step 4 ──
    DocumentGenerated    = "DocumentGenerated"     # 문서 생성 완료

    # ── Step 4 (HITL) → 분기 ──
    UserApproved         = "UserApproved"          # 사용자 승인 → Step 5
    UserRejected         = "UserRejected"          # 사용자 거부 → Step 2 재호출

    # ── Step 5 (Sign-off) → 종료 ──
    SignOffPassed        = "SignOffPassed"          # 최종 검토 통과
    SignOffReturned      = "SignOffReturned"        # 최종 검토 반려 → 재작업

    # ── 오류 처리 ──
    Error                = "Error"

print("이벤트 목록:")
for e in FnbProcessEvent:
    print("  -", e.value)

이벤트 목록:
  - Start
  - ContextGathered
  - AgentResultReady
  - DocumentGenerated
  - UserApproved
  - UserRejected
  - SignOffPassed
  - SignOffReturned
  - Error


---
## 2. Step 구현

> **`KernelProcessStep` 규칙**:
> - `@kernel_function`으로 장식된 메서드가 실행 단위
> - `KernelProcessStepContext`를 파라미터로 받으면 SK가 자동 주입
> - `await context.emit_event(FnbProcessEvent.XXX, data=...)` 로 다음 단계 트리거
> - 상태 유지가 필요하면 `KernelProcessStepState[StateModel]`로 선언

In [3]:
# Step 1: Context Gathering
# Extracts 5 F&B startup context items from user input
class ContextGatherStep(KernelProcessStep):
    """
    Extracts F&B startup context (business type, location, scale, stage, concern)
    from user input.

    Output event: FnbProcessEvent.ContextGathered
    Output data:  { business_type, location, scale, stage, concern, original_request }
    """

    @kernel_function(name='gather_context')
    async def gather_context(
        self,
        kernel: Kernel,
        user_input: str,
        context: KernelProcessStepContext,
    ) -> None:
        print('[Step 1] 컨텍스트 수집 시작...')

        # ⚠️ Azure RAI: prompt must be in English
        extraction_prompt = (
            'Extract F&B startup context from the following user request and return ONLY valid JSON.\n'
            '(No text outside JSON)\n\n'
            'Fields: business_type, location, scale, stage(preparing/pre-open/operating), concern\n'
            'Use null for missing fields.\n\n'
            'User request: %s\n\n'
            'JSON:'
        ) % user_input

        result = await kernel.invoke_prompt(extraction_prompt)
        raw = str(result).strip()

        try:
            ctx = json.loads(raw)
        except json.JSONDecodeError:
            ctx = {'business_type': None, 'location': None, 'scale': None, 'stage': None, 'concern': None}

        ctx['original_request'] = user_input
        print('[Step 1] 수집된 컨텍스트:', json.dumps(ctx, ensure_ascii=False))

        await context.emit_event(
            process_event=FnbProcessEvent.ContextGathered,
            data=ctx,
        )

print('Step 1 정의 완료')


Step 1 정의 완료


In [4]:
# Step 2: Agent Routing
class AgentRoutingStep(KernelProcessStep):
    """
    Analyzes context and delegates to the appropriate specialist agent.

    Input events:  FnbProcessEvent.ContextGathered, FnbProcessEvent.UserRejected
    Output event:  FnbProcessEvent.AgentResultReady
    """

    @kernel_function(name='route_to_agent')
    async def route_to_agent(
        self,
        kernel: Kernel,
        context_data: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        print('[Step 2] 에이전트 라우팅 시작...')

        user_request   = context_data.get('original_request', '')
        business_type  = context_data.get('business_type', 'general restaurant')
        location       = context_data.get('location', 'Seoul')

        # ⚠️ Azure RAI: routing prompt in English
        routing_prompt = (
            'Classify the following request into one of: legal, location, finance.\n'
            '- legal:    license, sanitation, tax, law\n'
            '- location: commercial area, foot traffic, competitors, rent\n'
            '- finance:  BEP, investment, revenue forecast, profit\n\n'
            'Request: %s\nType:'
        ) % user_request

        route_result = await kernel.invoke_prompt(routing_prompt)
        agent_type   = str(route_result).strip().lower()
        print('[Step 2] 라우팅 결과: %s' % agent_type)

        # ⚠️ Azure RAI: agent prompts in English, respond in Korean
        if 'legal' in agent_type:
            agent_prompt = (
                'You are an F&B legal and tax expert. '
                'Business type: %s, Location: %s. '
                'Question: %s '
                'Respond in Korean.'
            ) % (business_type, location, user_request)
        elif 'location' in agent_type:
            agent_prompt = (
                'You are an F&B commercial area analysis expert. '
                'Business type: %s, Location: %s. '
                'Question: %s '
                'Respond in Korean.'
            ) % (business_type, location, user_request)
        else:
            agent_prompt = (
                'You are an F&B startup finance analysis expert. '
                'Business type: %s, Location: %s. '
                'Question: %s '
                'Respond in Korean.'
            ) % (business_type, location, user_request)

        agent_result      = await kernel.invoke_prompt(agent_prompt)
        agent_result_text = str(agent_result)

        print('[Step 2] 에이전트 처리 완료 (%.50s...)' % agent_result_text)

        await step_context.emit_event(
            process_event=FnbProcessEvent.AgentResultReady,
            data={'agent_type': agent_type, 'result': agent_result_text, 'context': context_data},
        )

print('Step 2 정의 완료')


Step 2 정의 완료


In [5]:
# Step 3: Document Generation
class DocumentGenerationStep(KernelProcessStep):
    """
    Converts agent results into structured documents.
    (e.g. food business registration checklist, commercial area report, BEP report)

    Input event:  FnbProcessEvent.AgentResultReady
    Output event: FnbProcessEvent.DocumentGenerated
    """

    @kernel_function(name='generate_document')
    async def generate_document(
        self,
        kernel: Kernel,
        agent_output: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        print('[Step 3] 문서 생성 시작...')

        agent_type  = agent_output.get('agent_type', '')
        result_text = agent_output.get('result', '')
        ctx         = agent_output.get('context', {})

        # ⚠️ Azure RAI: document generation prompts in English
        if 'legal' in agent_type:
            doc_prompt = (
                'Format the following legal analysis as a Food Business Registration Checklist. '
                'Business type: %s, Location: %s. '
                'Write the document in Korean.\n\nContent:\n%s'
            ) % (ctx.get('business_type',''), ctx.get('location',''), result_text)
        elif 'location' in agent_type:
            doc_prompt = (
                'Format the following commercial area analysis as a Location Analysis Report. '
                'Write the document in Korean.\n\nContent:\n%s'
            ) % result_text
        else:
            doc_prompt = (
                'Format the following financial analysis as a BEP Analysis Report. '
                'Write the document in Korean.\n\nContent:\n%s'
            ) % result_text

        doc_result    = await kernel.invoke_prompt(doc_prompt)
        document_text = str(doc_result)

        print('[Step 3] 문서 생성 완료')

        await step_context.emit_event(
            process_event=FnbProcessEvent.DocumentGenerated,
            data={'document': document_text, 'agent_type': agent_type, 'original_result': result_text},
        )

print('Step 3 정의 완료')


Step 3 정의 완료


In [6]:
# ─────────────────────────────────────────────────────────
# Step 4: Human-in-the-Loop 게이트 (HITL)
#
# ⚠️ 중요: Python SK Process Framework의 HITL 지원은
#    2025년 현재 experimental 상태입니다.
#    실제 구현에서는 프로세스를 일시 중단하고 외부 이벤트로 재개합니다.
#    여기서는 자동 승인(시뮬레이션)으로 전체 흐름을 학습합니다.
# ─────────────────────────────────────────────────────────
class HumanReviewStep(KernelProcessStep):
    """
    문서를 사용자에게 제시하고 승인/거부를 기다립니다.
    
    [실제 구현 방식]
    1. 문서를 외부 pub/sub 시스템(Azure Service Bus 등)으로 발행
    2. 프로세스 일시 중단 (idle state)
    3. 사용자가 UI에서 승인/거부 → 외부에서 이벤트 주입
    4. 프로세스 재개
    
    [이 노트북에서는 자동 승인으로 시뮬레이션]
    
    입력 이벤트: FnbProcessEvent.DocumentGenerated
    출력 이벤트: FnbProcessEvent.UserApproved 또는 FnbProcessEvent.UserRejected
    """

    # 자동 승인 시뮬레이션 여부 (실제 구현에서는 False)
    AUTO_APPROVE: ClassVar[bool] = True

    @kernel_function(name="request_human_review")
    async def request_human_review(
        self,
        doc_data: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        document = doc_data.get("document", "")

        print("\n" + "="*60)
        print("[Step 4: HITL 게이트] 사용자 검토 요청")
        print("-"*60)
        print("📄 생성된 문서 미리보기:")
        print(document[:500] + ("..." if len(document) > 500 else ""))
        print("="*60)

        if self.AUTO_APPROVE:
            # 시뮬레이션: 자동 승인
            print("[Step 4] ✅ 자동 승인 (AUTO_APPROVE=True)")
            await step_context.emit_event(
                process_event=FnbProcessEvent.UserApproved,
                data=doc_data,
            )
        else:
            # 실제 구현:
            # 1. 외부 큐에 승인 요청 발행
            # 2. 프로세스 일시 중단 (아무 이벤트도 emit하지 않음)
            # 3. 외부 시스템이 UserApproved/UserRejected 이벤트를 주입
            print("[Step 4] ⏸️ 프로세스 일시 중단 — 사용자 응답 대기")
            # 실제 구현 예시:
            # await publish_to_service_bus("review-requests", doc_data)
            # (프로세스는 여기서 idle 상태로 대기)

print("Step 4 정의 완료")

Step 4 정의 완료


In [7]:
# Step 5: Sign-off Agent (Final Review)
class SignOffStep(KernelProcessStep):
    """
    Reviews the document and decides PASS or RETURN.

    Review checklist:
    1. disclaimer_present  - disclaimer included
    2. scope_violation     - out of F&B scope
    3. definitive_expression - absolute expressions used
    4. law_consistency     - obvious legal errors
    5. personal_info_leak  - personal info exposed
    6. missing_source      - missing data source citations
    7. overall             - PASS or RETURN

    Input event:  FnbProcessEvent.UserApproved
    Output event: FnbProcessEvent.SignOffPassed or FnbProcessEvent.SignOffReturned
    """

    @kernel_function(name='sign_off_review')
    async def sign_off_review(
        self,
        kernel: Kernel,
        approved_data: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        print('\n[Step 5: Sign-off] 최종 검토 시작...')

        document = approved_data.get('document', '')

        # ⚠️ Azure RAI: review prompt in English
        review_prompt = (
            'You are the final review agent for an F&B startup AI system. '
            'Review the document below and return ONLY valid JSON (no markdown).\n\n'
            'Review fields:\n'
            '1. disclaimer_present: is a disclaimer included? (true/false)\n'
            '2. scope_violation: is there content outside F&B startup scope? (true/false)\n'
            '3. definitive_expression: are absolute expressions used? (true/false)\n'
            '4. law_consistency: are there obvious legal errors? (true/false)\n'
            '5. personal_info_leak: is personal info exposed? (true/false)\n'
            '6. missing_source: are data sources missing? (true/false)\n'
            '7. overall: "PASS" or "RETURN"\n'
            '8. reason: reason for decision (max 50 chars)\n\n'
            'Document to review:\n%s\n\n'
            'Return JSON only:'
        ) % document

        review_result = await kernel.invoke_prompt(review_prompt)
        raw = str(review_result).strip().replace('```json','').replace('```','')

        try:
            review = json.loads(raw)
        except json.JSONDecodeError:
            review = {'overall': 'RETURN', 'reason': 'Sign-off 파싱 오류'}

        overall = review.get('overall', 'RETURN').upper()
        reason  = review.get('reason', '')

        print('[Step 5] Sign-off 결과: %s — %s' % (overall, reason))
        print('[Step 5] 검토 상세:', json.dumps(review, ensure_ascii=False))

        if overall == 'PASS':
            print('\n✅ [PASS] 최종 검토 통과! 사용자에게 전달합니다.')
            await step_context.emit_event(
                process_event=FnbProcessEvent.SignOffPassed,
                data={'final_document': document, 'review': review},
            )
        else:
            print('\n🔄 [RETURN] 수정 필요 — 오케스트레이터에 재요청합니다.')
            await step_context.emit_event(
                process_event=FnbProcessEvent.SignOffReturned,
                data={'document': document, 'review': review, 'reason': reason},
            )

print('Step 5 (Sign-off) 정의 완료')


Step 5 (Sign-off) 정의 완료


---
## 3. Process 조립 — ProcessBuilder로 Step 연결

In [8]:
def build_fnb_process():
    """
    F&B 창업 지원 프로세스를 조립합니다.
    
    [단계 연결도]
    Start
      → ContextGatherStep.gather_context
      → (ContextGathered) → AgentRoutingStep.route_to_agent
      → (AgentResultReady) → DocumentGenerationStep.generate_document
      → (DocumentGenerated) → HumanReviewStep.request_human_review
      → (UserApproved) → SignOffStep.sign_off_review
      → (SignOffPassed) → 프로세스 종료
      → (SignOffReturned) → AgentRoutingStep (재작업)
      → (UserRejected) → AgentRoutingStep (재작업)
    """
    builder = ProcessBuilder(name="FnbStartupAssistantProcess")

    # ── 단계 등록 ──
    step_context_gather = builder.add_step(ContextGatherStep)
    step_routing        = builder.add_step(AgentRoutingStep)
    step_doc_gen        = builder.add_step(DocumentGenerationStep)
    step_human_review   = builder.add_step(HumanReviewStep)
    step_sign_off       = builder.add_step(SignOffStep)

    # ── 이벤트 연결 ──

    # 진입점: Start 이벤트 → ContextGatherStep
    builder.on_input_event(FnbProcessEvent.Start) \
        .send_event_to(step_context_gather, function_name="gather_context", parameter_name="user_input")

    # Step 1 완료 → Step 2
    step_context_gather.on_event(FnbProcessEvent.ContextGathered) \
        .send_event_to(step_routing, function_name="route_to_agent", parameter_name="context_data")

    # Step 2 완료 → Step 3
    step_routing.on_event(FnbProcessEvent.AgentResultReady) \
        .send_event_to(step_doc_gen, function_name="generate_document", parameter_name="agent_output")

    # Step 3 완료 → Step 4 (HITL)
    step_doc_gen.on_event(FnbProcessEvent.DocumentGenerated) \
        .send_event_to(step_human_review, function_name="request_human_review", parameter_name="doc_data")

    # Step 4 승인 → Step 5 (Sign-off)
    step_human_review.on_event(FnbProcessEvent.UserApproved) \
        .send_event_to(step_sign_off, function_name="sign_off_review", parameter_name="approved_data")

    # Step 4 거부 → Step 2 재실행
    step_human_review.on_event(FnbProcessEvent.UserRejected) \
        .send_event_to(step_routing, function_name="route_to_agent", parameter_name="context_data")

    # Step 5 RETURN → Step 2 재실행
    step_sign_off.on_event(FnbProcessEvent.SignOffReturned) \
        .send_event_to(step_routing, function_name="route_to_agent", parameter_name="context_data")

    return builder.build()


fnb_process = build_fnb_process()
print("✅ F&B 창업 지원 프로세스 조립 완료!")

✅ F&B 창업 지원 프로세스 조립 완료!


---
## 4. 프로세스 실행

In [9]:
kernel = make_kernel()

# ⚠️ Azure RAI: initial event data must be in English
await start(
    process=fnb_process,
    kernel=kernel,
    initial_event=KernelProcessEvent(
        id=FnbProcessEvent.Start,
        data=(
            'I am planning to open a 30-pyeong general restaurant '
            'near Hongdae in Mapo-gu, Seoul. '
            'Please summarize the sanitation license procedures and required documents. '
            'Respond in Korean.'
        ),
    ),
)

print('\n🎉 프로세스 실행 완료!')


[Step 1] 컨텍스트 수집 시작...
[Step 1] 수집된 컨텍스트: {"business_type": null, "location": null, "scale": null, "stage": null, "concern": null, "original_request": "I am planning to open a 30-pyeong general restaurant near Hongdae in Mapo-gu, Seoul. Please summarize the sanitation license procedures and required documents. Respond in Korean."}
[Step 2] 에이전트 라우팅 시작...
[Step 2] 라우팅 결과: legal
[Step 2] 에이전트 처리 완료 (홍대 인근 마포구에서 30평 규모의 일반 음식점을 계획하시고 계시다면, 위생 관련 인허가 ...)
[Step 3] 문서 생성 시작...
[Step 3] 문서 생성 완료

[Step 4: HITL 게이트] 사용자 검토 요청
------------------------------------------------------------
📄 생성된 문서 미리보기:
### 음식점 영업신고 체크리스트

**비즈니스 유형:** 없음  
**위치:** 없음

---

홍대 인근 마포구에서 30평 규모의 일반 음식점을 계획하시고 계시다면, 위생 관련 인허가 절차와 필요 서류에 대해 안내드리겠습니다.

1. **영업 신고**
   - 일반 음식점을 운영하기 위해서는 반드시 영업 신고가 필요합니다. 이는 마포구청의 보건소에서 처리할 수 있습니다.

2. **시설 기준 충족**
   - 음식점 영업을 위한 필수적인 시설 기준을 충족해야 합니다. 주방, 위생 설비, 환기 시설 등 관련 기준을 점검하시기 바랍니다.

3. **필요 서류**
   - 영업신고서 (구청에서 제공)
   - 사업자 등록증 사본
   - 임대차계약서 또는 건물 소유 증명서
   - 조리사의 건강진단결과서 (

In [10]:
# 재무 분석 테스트
await start(
    process=fnb_process,
    kernel=kernel,
    initial_event=KernelProcessEvent(
        id=FnbProcessEvent.Start,
        data=(
            'I am planning to open a cafe. '
            'Initial investment: 80,000,000 KRW, monthly fixed cost: 3,500,000 KRW, '
            'avg revenue per customer: 7,000 KRW, variable cost ratio: 35%. '
            'Please calculate the break-even point. Respond in Korean.'
        ),
    ),
)
print('\n🎉 재무 분석 프로세스 완료!')


[Step 1] 컨텍스트 수집 시작...
[Step 1] 수집된 컨텍스트: {"business_type": null, "location": null, "scale": null, "stage": null, "concern": null, "original_request": "I am planning to open a cafe. Initial investment: 80,000,000 KRW, monthly fixed cost: 3,500,000 KRW, avg revenue per customer: 7,000 KRW, variable cost ratio: 35%. Please calculate the break-even point. Respond in Korean."}
[Step 2] 에이전트 라우팅 시작...
[Step 2] 라우팅 결과: finance
[Step 2] 에이전트 처리 완료 (카페를 여는 것을 계획 중이시군요. 말씀하신 초기 투자금과 비용을 토대로 손익분기점을 계산...)
[Step 3] 문서 생성 시작...
[Step 3] 문서 생성 완료

[Step 4: HITL 게이트] 사용자 검토 요청
------------------------------------------------------------
📄 생성된 문서 미리보기:
# 손익분기점(BEP) 분석 보고서

## 개요

카페를 여는 계획을 세우고 계신 것에 대해 환영합니다. 본 보고서는 귀하가 제공한 초기 투자금 및 비용 정보를 기반으로 손익분기점을 계산한 결과를 담고 있습니다. 손익분기점은 사업의 수익이 고정 비용을 초과하는 지점을 나타내며, 이는 사업의 안정성과 지속 가능성을 평가하는 중요한 지표입니다.

## 초기 투자 및 비용

- **초기 투자**: 80,000,000 KRW
- **월 고정 비용**: 3,500,000 KRW
- **고객당 평균 수익**: 7,000 KRW
- **변동 비용 비율**: 35%

## 손익분기점 계산

손익분기점을 계산하기 위해선 고정 비용을 이익률로 

---
## 5. ✅ 전체 학습 시리즈 정리

```
STEP 01 — Kernel & Plugin
  @kernel_function으로 Python 함수를 LLM 툴로 등록
  FunctionChoiceBehavior.Auto() 로 자동 호출

STEP 02 — ChatCompletionAgent
  역할·지시문·플러그인을 가진 전문 에이전트 구현
  ChatHistoryAgentThread로 멀티턴 대화 세션 관리

STEP 03 — AgentGroupChat
  KernelFunctionSelectionStrategy: LLM이 다음 에이전트 선택
  KernelFunctionTerminationStrategy: 종료 조건 판단
  트리아지 패턴 + Plugin-as-Agent 패턴

STEP 04 — MCP 연동
  MCP 서버로 카카오 지도 API, 법령 API 래핑
  McpServerPlugin으로 SK 에이전트에 연결

STEP 05 — Process Framework + HITL + Sign-off
  ProcessBuilder로 5단계 워크플로우 구성
  HITL 게이트 (일시 중단 → 사용자 승인 → 재개)
  Sign-off Agent의 PASS / RETURN 분기
```

---

### 다음 단계: 프로덕션 고려사항

| 항목 | 현재 (PoC) | 프로덕션 |
|------|-----------|----------|
| 상태 저장 | InMemory | Azure Cosmos DB / Redis |
| HITL 게이트 | 자동 승인 (시뮬레이션) | Azure Service Bus + 웹훅 |
| 프로세스 배포 | Local Runtime | Dapr 분산 런타임 |
| API 서버 | - | FastAPI + StreamingResponse |
| 프론트엔드 | - | React + Server-Sent Events |